# 09 — Serie temporali IV: STL (1990) e il ponte NDSharp
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

**STL** (Cleveland, Cleveland, McRae & Terpenning, 1990) separa trend-ciclo, stagionalità e resto con LOESS. Si decompone prima, poi si prevede ogni componente per conto suo.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Stagionalità che cresce nel tempo, trend non lineare, due outlier.
var rng = new Random(1990);
const int m = 12;
var n = 144;
var trendTrue = Enumerable.Range(0, n).Select(t => 50 + 10 * Math.Tanh((t - 70) / 30.0)).ToArray();
var seasonTrue = Enumerable.Range(0, n).Select(t => (3 + t / 48.0) * Math.Sin(2 * Math.PI * t / m)).ToArray();
var y = Enumerable.Range(0, n).Select(t => trendTrue[t] + seasonTrue[t] + 0.7 * Simulation.StandardNormal(rng)).ToArray();
y[40] += 15; y[100] -= 12;
Chart.Lines("Serie osservata (outlier a t = 40 e t = 100)", new Line("y", y));

## L'algoritmo
**LOESS** locale lineare: in ogni punto si stima una retta pesata sui $q$ vicini, con pesi tricubici $(1-(d/h)^3)^3$. Una retta viene riprodotta esattamente, anche quando si estrapola.

**Ciclo interno**, ripetuto $n_i$ volte:
1. detrend: $y - T$;
2. smoothing delle **sottoserie cicliche**: tutti i gennaio, tutti i febbraio, … ciascuno esteso di un ciclo prima e dopo;
3. **passa-basso** del risultato: MA($n_p$), MA($n_p$), MA(3), poi LOESS($n_l$);
4. $S = C - L$;
5. destagionalizza;
6. LOESS del trend con $n_t \ge 1.5n_p/(1-1.5/n_s)$.

**Ciclo esterno** (robustezza): pesi bisquare $B(|R_t|/6\,\mathrm{med}|R|)$ che declassano gli outlier nelle iterazioni successive.

Proprietà verificata dai test: su retta + stagionalità periodica a media nulla, STL separa le componenti **esattamente**. I LOESS riproducono le rette, e una media mobile lunga un periodo annulla la stagionalità.

In [ ]:
var plain = Stl.Decompose(y, m, seasonalSpan: 13);
var robust = Stl.Decompose(y, m, seasonalSpan: 13, innerIterations: 1, robustIterations: 6);
Chart.Panels("STL robusta: componenti", new Line("osservato", y), new Line("trend", robust.Trend), new Line("stagionalità", robust.Seasonal), new Line("resto", robust.Remainder));
double Rmse(double[] a, double[] b) => Accuracy.Rmse(a, b);
Table.Show(new[] { "STL", "RMSE trend vs vero", "RMSE stagionalità vs vera", "peso robusto t=40", "peso t=100" }, new[]
{
    new object[] { "non robusta", Rmse(plain.Trend, trendTrue), Rmse(plain.Seasonal, seasonTrue), plain.RobustnessWeights[40], plain.RobustnessWeights[100] },
    new object[] { "robusta (6 iterazioni)", Rmse(robust.Trend, trendTrue), Rmse(robust.Seasonal, seasonTrue), robust.RobustnessWeights[40], robust.RobustnessWeights[100] },
});

## Decomporre prima, prevedere le componenti poi
- Trend destagionalizzato: Holt con damping.
- Stagionalità: seasonal naive applicata alla sola componente stagionale (l'ultimo ciclo stimato si ripete).
- Previsione = somma delle due.

Il confronto è con Holt-Winters e con il seasonal naive sulla serie grezza.

In [ ]:
double[] train = y[..132], test = y[132..];
var stl = Stl.Decompose(train, m, seasonalSpan: 13, innerIterations: 1, robustIterations: 6);
var seasonallyAdjusted = train.Select((v, t) => v - stl.Seasonal[t]).ToArray();
var trendF = ExponentialSmoothing.Holt(seasonallyAdjusted, 0.3, 0.1, 12, phi: 0.9).Forecast;
var seasonF = ReferenceForecasts.SeasonalNaive(stl.Seasonal, m, 12);
var stlF = trendF.Select((v, h) => v + seasonF[h]).ToArray();
var hwF = ExponentialSmoothing.HoltWinters(train, m, 0.3, 0.05, 0.2, 12, phi: 0.9).Forecast;
var snF = ReferenceForecasts.SeasonalNaive(train, m, 12);
Table.Show(new[] { "Metodo", "MASE (m=12)" }, new[]
{
    new object[] { "STL + damped Holt + seasonal naive", Accuracy.Mase(test, stlF, train, m) },
    new object[] { "Holt-Winters damped", Accuracy.Mase(test, hwF, train, m) },
    new object[] { "Seasonal naive", Accuracy.Mase(test, snF, train, m) },
});
Chart.Lines("Ultimi 3 anni e previsioni (asse x: t − 108)", new Line("osservato", y[108..]), new Line("STL + componenti", stlF, Start: 24, Dashed: true), new Line("Holt-Winters", hwF, Start: 24, Dashed: true));

## Il ponte NDSharp
`NDSharp` (dal repository PySharp) è un ndarray con semantica NumPy: dtype a runtime, viste strided, broadcasting. `Numbers` resta tipizzato, con Generic Math e `Span<T>`. Il pacchetto `MathemaSharp.NDSharp` converte fra i due mondi:
- `ToVector()` e `ToMatrix()` copiano in ordine C con cast del dtype;
- `ToNDArray()` avvolge un `double[]` **senza copia**.

Il flusso tipico: preparazione dei dati con NDSharp, algoritmo con MathemaSharp.

In [ ]:
#load "../shared/NDSharp.csx"
using NDSharp;
using MathemaSharp.NDSharp;
// Preparazione "alla NumPy": rampa + seno costruiti con le ufunc di NDSharp, poi STL di MathemaSharp.
var t = np.Arange(0L, 120L);
var series = np.Add(np.Multiply(t, NDArray.Scalar(0.1)), np.Sin(np.Multiply(t, NDArray.Scalar(2 * Math.PI / 12))));
Console.WriteLine($"NDArray dtype={series.DType}, shape=({string.Join(",", series.Shape)})");
var decomposition = Stl.Decompose(series.ToVector(), 12);
var trendNd = decomposition.Trend.ToNDArray();
Console.WriteLine($"Trend come NDArray: dtype={trendNd.DType}, pendenza media={(decomposition.Trend[^1] - decomposition.Trend[0]) / 119:F6} (vera 0.1)");

## Esperimenti
- `seasonalSpan` più grande rende la stagionalità più rigida: provare 7 e 35 e guardare la componente stagionale.
- Senza robustezza, gli outlier finiscono nella stagionalità di *tutti* i mesi omologhi: perché?
- Sostituire il seasonal naive sulla componente stagionale con Holt-Winters a γ piccolo.